# Web Scraping Exercise

Web Scraping allows you to gather large volumes of data from diverse and real-time online sources. This data can be crucial for enriching your datasets, filling in gaps, and providing current information that enhances the quality and relevance of your analysis. Web scraping enables you to collect data that might not be readily available through traditional APIs or databases, offering a competitive edge by incorporating unique and comprehensive insights. Moreover, it automates the data collection process, saving time and resources while ensuring a scalable approach to continuously updating and maintaining your datasets.

Ethical web scraping involves respecting website terms of service, avoiding overloading servers, and ensuring that the collected data is used responsibly and in compliance with privacy laws and regulations.

Use Python, ```requests```, ```BeautifulSoup``` and/or ```pandas``` to scrape web data:

## Import Libraries

In [1]:
from pathlib import Path
import requests
import pandas as pd
from bs4 import BeautifulSoup

## Define the Target URL

In [2]:
url = "https://news.google.com/rss/search?q=Tesla%20stock&hl=en-US&gl=US&ceid=US:en"

## Send a Request to the Website

Do not forget to check the response status code

In [3]:
headers = {
    "User-Agent": "Mozilla/5.0"
}

response = requests.get(url, headers=headers, timeout=10)

print("Status Code:", response.status_code)

if response.status_code == 200:
    print("Request successful.")
else:
    print("Request failed.")

Status Code: 200
Request successful.


## Parse the HTML Content

Use a library to access the HTMl content

In [4]:
soup = BeautifulSoup(response.content, "html.parser")

items = soup.find_all("item")

print("Number of Tesla news articles found:", len(items))

Number of Tesla news articles found: 100


C:\Users\noahf\AppData\Local\Temp\ipykernel_11332\1524488176.py:1: XMLParsedAsHTMLWarning: It looks like you're using an HTML parser to parse an XML document.

Assuming this really is an XML document, what you're doing might work, but you should know that using an XML parser will be more reliable. To parse this document as XML, make sure you have the Python package 'lxml' installed, and pass the keyword argument `features="xml"` into the BeautifulSoup constructor.

If you want or need to use an HTML parser on this document, you can make this warning go away by filtering it. To do that, run this code before calling the BeautifulSoup constructor:

    from bs4 import XMLParsedAsHTMLWarning
    import warnings

    warnings.filterwarnings("ignore", category=XMLParsedAsHTMLWarning)

  soup = BeautifulSoup(response.content, "html.parser")


## Identify the Data to be Scraped

Write a couple of sentence on the data you want to scrape

I want to scrape recent Tesla stock-related news from Google News RSS.  
The collected data includes article titles, publication dates, source names and links.  
This data is useful because it can later be compared with existing Tesla stock price data.

## Extract Data

Find specific elements and extract text or attributes from elements (handle pagination if necessary)

In [5]:
news_data = []

for item in items:
    title = item.find("title").text if item.find("title") else None
    link = item.find("link").text if item.find("link") else None
    pub_date = item.find("pubdate").text if item.find("pubdate") else None
    source = item.find("source").text if item.find("source") else None
    
    news_data.append({
        "title": title,
        "published_at": pub_date,
        "source": source,
        "link": link
    })

scraped_df = pd.DataFrame(news_data)

scraped_df.head()

,title,published_at,source,link
0,Tesla Stock Drops as CFO Sells Shares. Can It ...,"Mon, 18 May 2026 15:47:00 GMT",,
1,Tesla Raises U.S. Model Y Prices. Its Stock Ju...,"Mon, 18 May 2026 10:34:00 GMT",,
2,Man who bought world's first electric sports c...,"Sun, 17 May 2026 18:05:00 GMT",,
3,Why Tesla stock is down around 2% on Monday - ...,"Mon, 18 May 2026 15:27:11 GMT",,
4,Tesla's stock surrenders gains after earnings....,"Wed, 22 Apr 2026 07:00:00 GMT",,


## Store Data in a Structured Format

Give a brief overview of the data collected (e.g. count, fields, ...)

In [6]:
scraped_df["published_at"] = pd.to_datetime(scraped_df["published_at"], errors="coerce", utc=True)
scraped_df["date"] = scraped_df["published_at"].dt.date

scraped_df = scraped_df.dropna(subset=["title", "published_at"])
scraped_df = scraped_df.drop_duplicates(subset=["title"])
scraped_df = scraped_df.sort_values("published_at", ascending=False)

print("Number of Tesla news articles:", len(scraped_df))
print("Columns:", list(scraped_df.columns))

scraped_df.head()

Number of Tesla news articles: 71
Columns: ['title', 'published_at', 'source', 'link', 'date']


,title,published_at,source,link,date
25,Lawsuit Over Full Self-Driving Promises Succee...,2026-05-18 17:06:02+00:00,,,2026-05-18
19,Tesla’s Elon Musk Finds Time to Update Us on t...,2026-05-18 16:02:00+00:00,,,2026-05-18
13,Why Tesla stock is down around 2% on Monday - ...,2026-05-18 16:00:06+00:00,,,2026-05-18
0,Tesla Stock Drops as CFO Sells Shares. Can It ...,2026-05-18 15:47:00+00:00,,,2026-05-18
23,"Micron, Regeneron, UnitedHealth, Dominion Ener...",2026-05-18 15:38:00+00:00,,,2026-05-18


## Save the Data

In [7]:
project_root = Path.cwd().parent

processed_dir = project_root / "data" / "processed"
processed_dir.mkdir(parents=True, exist_ok=True)

output_path = processed_dir / "tesla_news_cleaned.csv"

scraped_df.to_csv(output_path, index=False)

print("Saved file to:", output_path)

Saved file to: c:\Users\noahf\Documents\Technikum BWI\4. Semester\BDE\BDE-financial-news-stock-project\data\processed\tesla_news_cleaned.csv


## Check if file exists

In [8]:
check_path = processed_dir / "tesla_news_cleaned.csv"

print("File exists:", check_path.exists())

check_df = pd.read_csv(check_path)
print("Rows:", len(check_df))
check_df.head()

File exists: True
Rows: 71


,title,published_at,source,link,date
0,Lawsuit Over Full Self-Driving Promises Succee...,2026-05-18 17:06:02+00:00,NaN,NaN,2026-05-18
1,Tesla’s Elon Musk Finds Time to Update Us on t...,2026-05-18 16:02:00+00:00,NaN,NaN,2026-05-18
2,Why Tesla stock is down around 2% on Monday - ...,2026-05-18 16:00:06+00:00,NaN,NaN,2026-05-18
3,Tesla Stock Drops as CFO Sells Shares. Can It ...,2026-05-18 15:47:00+00:00,NaN,NaN,2026-05-18
4,"Micron, Regeneron, UnitedHealth, Dominion Ener...",2026-05-18 15:38:00+00:00,NaN,NaN,2026-05-18
